# Démo : de la transformation à l'action

Les trajets des taxis jaunes de New York, janvier 2024, publiés par la Taxi and Limousine Commission (TLC).
On lit un fichier Parquet et un fichier CSV, on écrit des transformations, puis on lance des actions. Pendant
la démo, on suit le travail du cluster dans la Spark UI : http://localhost:4040, onglet **Jobs**.

**Critère de réussite** : les transformations s'écrivent sans aucun calcul ; le nombre de trajets, total et
par type de paiement, calculé par Spark sur le cluster, est identique au calcul de pandas sur le même fichier.
La dernière cellule le vérifie et s'arrête en erreur sinon.

Avant de commencer : le cluster tourne (`docker compose up -d` à la racine du module), et ce notebook est
ouvert depuis son dossier, avec le noyau `.venv` du module (`uv sync --group tp`).

## Préparation

Une cellule qui commence par `%%client` contient des commandes bash, exécutées dans le conteneur client du
cluster, où est installé `hdfs`.

In [1]:
import os
import subprocess
from IPython.core.magic import register_cell_magic

os.environ["GRPC_ENABLE_FORK_SUPPORT"] = "false"   # pas de message de gRPC quand %%client lance docker


@register_cell_magic
def client(ligne, cellule):
    """%%client : les commandes de la cellule s'exécutent dans le conteneur client du cluster."""
    commande = ["docker", "compose", "exec", "-T", "client", "bash", "-c", "set -e\n" + cellule]
    resultat = subprocess.run(commande, capture_output=True, text=True)
    print(resultat.stdout + resultat.stderr, end="")
    if resultat.returncode != 0:
        raise RuntimeError("La commande a échoué : voir sa sortie ci-dessus")

Télécharger les trajets de janvier 2024 (50 Mo) et la table des zones de taxi, s'ils sont absents.

In [2]:
import urllib.request

TLC = "https://d37ci6vzurychx.cloudfront.net"
FICHIERS = {"yellow_tripdata_2024-01.parquet": f"{TLC}/trip-data/yellow_tripdata_2024-01.parquet",
            "taxi_zone_lookup.csv": f"{TLC}/misc/taxi_zone_lookup.csv"}
os.makedirs("data", exist_ok=True)
for nom, url in FICHIERS.items():
    if not os.path.exists(f"data/{nom}"):
        urllib.request.urlretrieve(url, f"data/{nom}")
    print(nom, os.path.getsize(f"data/{nom}"), "octets")

yellow_tripdata_2024-01.parquet 49961641 octets
taxi_zone_lookup.csv 12331 octets


Copier les deux fichiers dans le conteneur client, puis les déposer dans un dossier `/demo4` vide de HDFS.

In [3]:
for nom in FICHIERS:
    subprocess.run(["docker", "compose", "cp", f"data/{nom}", f"client:/tmp/{nom}"], check=True,
                   capture_output=True)

In [4]:
%%client
hdfs dfs -rm -r -f -skipTrash /demo4
hdfs dfs -mkdir -p /demo4
hdfs dfs -put /tmp/yellow_tripdata_2024-01.parquet /demo4/janvier.parquet
hdfs dfs -put /tmp/taxi_zone_lookup.csv /demo4/zones.csv
hdfs dfs -ls /demo4

Deleted /demo4
Found 2 items
-rw-r--r--   3 spark supergroup   49961641 2026-11-26 14:18 /demo4/janvier.parquet
-rw-r--r--   3 spark supergroup      12331 2026-11-26 14:18 /demo4/zones.csv


## 1. La session

Le notebook se connecte au serveur Spark Connect du cluster (port 15002). Le driver tourne dans ce serveur,
les executors sur les workers.

In [5]:
from pyspark.sql import SparkSession, functions as F

spark = SparkSession.builder.remote("sc://localhost:15002").getOrCreate()
print("Spark", spark.version)

Spark 4.1.3


## 2. Lire un fichier Parquet

Le schéma est écrit dans le fichier : Spark le lit sans rien deviner.

In [6]:
trajets = spark.read.parquet("/demo4/janvier.parquet")
trajets.printSchema()

root
 |-- VendorID: integer (nullable = true)
 |-- tpep_pickup_datetime: timestamp_ntz (nullable = true)
 |-- tpep_dropoff_datetime: timestamp_ntz (nullable = true)
 |-- passenger_count: long (nullable = true)
 |-- trip_distance: double (nullable = true)
 |-- RatecodeID: long (nullable = true)
 |-- store_and_fwd_flag: string (nullable = true)
 |-- PULocationID: integer (nullable = true)
 |-- DOLocationID: integer (nullable = true)
 |-- payment_type: long (nullable = true)
 |-- fare_amount: double (nullable = true)
 |-- extra: double (nullable = true)
 |-- mta_tax: double (nullable = true)
 |-- tip_amount: double (nullable = true)
 |-- tolls_amount: double (nullable = true)
 |-- improvement_surcharge: double (nullable = true)
 |-- total_amount: double (nullable = true)
 |-- congestion_surcharge: double (nullable = true)
 |-- Airport_fee: double (nullable = true)



## 3. Des transformations : rien n'est calculé

Notez le nombre de jobs dans l'onglet Jobs, puis exécutez la cellule : il ne change pas.

In [7]:
%%time
valides = trajets.filter((F.col("trip_distance") > 0) & (F.col("total_amount") > 0))
colonnes = valides.select("payment_type", "trip_distance", "total_amount", "PULocationID")
print("Plan enregistré, aucun calcul")

Plan enregistré, aucun calcul
CPU times: user 615 μs, sys: 221 μs, total: 836 μs
Wall time: 1.92 ms


## 4. Une action : le calcul part

`count()` est une action. Onglet Jobs : de nouveaux jobs apparaissent.

In [8]:
%%time
n_spark = colonnes.count()
print(n_spark, "trajets retenus")

2872021 trajets retenus
CPU times: user 4.85 ms, sys: 6.16 ms, total: 11 ms
Wall time: 373 ms


In [9]:
colonnes.show(5)

+------------+-------------+------------+------------+
|payment_type|trip_distance|total_amount|PULocationID|
+------------+-------------+------------+------------+
|           2|         1.72|        22.7|         186|
|           1|          1.8|       18.75|         140|
|           1|          4.7|        31.3|         236|
|           1|          1.4|        17.0|          79|
|           1|          0.8|        16.1|         211|
+------------+-------------+------------+------------+
only showing top 5 rows


## 5. Une deuxième action relance tout le plan

Spark n'a rien gardé du `count()` : il relit le fichier et refait le filtre.

In [10]:
%%time
total = colonnes.agg(F.sum("total_amount")).first()[0]
print(round(total), "dollars")

78488879 dollars
CPU times: user 4.82 ms, sys: 3.48 ms, total: 8.29 ms
Wall time: 225 ms


## 6. Regrouper : les trajets par type de paiement

In [11]:
par_paiement = colonnes.groupBy("payment_type").count().orderBy("payment_type")
par_paiement.show()

+------------+-------+
|payment_type|  count|
+------------+-------+
|           0| 117329|
|           1|2298408|
|           2| 422863|
|           3|  10615|
|           4|  22806|
+------------+-------+



## 7. Lire un fichier CSV

Sans option, Spark ne connaît ni le nom ni le type des colonnes.

In [12]:
spark.read.csv("/demo4/zones.csv").printSchema()

root
 |-- _c0: string (nullable = true)
 |-- _c1: string (nullable = true)
 |-- _c2: string (nullable = true)
 |-- _c3: string (nullable = true)



Avec `header`, la première ligne donne les noms ; avec `inferSchema`, Spark parcourt le fichier pour deviner les types.

In [13]:
zones = (spark.read
         .option("header", True)
         .option("inferSchema", True)
         .csv("/demo4/zones.csv"))
zones.printSchema()
zones.show(3)

root
 |-- LocationID: integer (nullable = true)
 |-- Borough: string (nullable = true)
 |-- Zone: string (nullable = true)
 |-- service_zone: string (nullable = true)

+----------+-------+--------------------+------------+
|LocationID|Borough|                Zone|service_zone|
+----------+-------+--------------------+------------+
|         1|    EWR|      Newark Airport|         EWR|
|         2| Queens|         Jamaica Bay|   Boro Zone|
|         3|  Bronx|Allerton/Pelham G...|   Boro Zone|
+----------+-------+--------------------+------------+
only showing top 3 rows


## 8. Écrire le résultat en Parquet

L'écriture est une action. Le résultat est un dossier, pas un fichier.

In [14]:
par_paiement.write.mode("overwrite").parquet("/demo4/par_paiement")

In [15]:
%%client
hdfs dfs -ls /demo4/par_paiement

Found 2 items
-rw-r--r--   3 spark supergroup          0 2026-11-26 14:18 /demo4/par_paiement/_SUCCESS
-rw-r--r--   3 spark supergroup        808 2026-11-26 14:18 /demo4/par_paiement/part-00000-1fd6c7d7-0a54-40d9-9c20-4d1dc1a55e3a-c000.snappy.parquet


## Le critère de réussite

pandas lit le même fichier sur cette machine, applique le même filtre et compte.

In [16]:
import pandas as pd

local = pd.read_parquet("data/yellow_tripdata_2024-01.parquet",
                        columns=["payment_type", "trip_distance", "total_amount"])
local = local[(local["trip_distance"] > 0) & (local["total_amount"] > 0)]
paiements_pandas = local["payment_type"].value_counts().to_dict()
paiements_spark = {ligne["payment_type"]: ligne["count"] for ligne in par_paiement.collect()}
print("Même nombre de trajets :", len(local) == n_spark)
print("Mêmes trajets par type de paiement :", paiements_pandas == paiements_spark)
assert len(local) == n_spark and paiements_pandas == paiements_spark
print("Critère de réussite atteint")

Même nombre de trajets : True
Mêmes trajets par type de paiement : True
Critère de réussite atteint
